In [ ]:
# Setup: in Google Colab, download and install the repository (a few minutes, once per session); elsewhere, work from the
# repository. Pages that read the derived catalogues need Zenodo bundle 1 (python scripts/fetch_release.py).
import os, subprocess, sys
os.environ['MPLBACKEND'] = 'Agg'   # as in a plain script run: Jupyter's inline settings would change the saved figures
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/lmc-ancient-disc'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vasilybelokurov/lmc-ancient-disc', '/content/lmc-ancient-disc'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/lmc-ancient-disc'], check=True)
    os.chdir('/content/lmc-ancient-disc')
else:
    while not os.path.exists('pyproject.toml'):
        os.chdir('..')
sys.path[:0] = ['src', 'scripts', 'kb']
if not os.path.exists('data/giants_apogee.parquet'):   # the catalogues of Zenodo bundle 1, checked against data/sources.yaml
    subprocess.run([sys.executable, 'scripts/fetch_release.py'], check=True)

# The most metal-poor stars

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/lmc-ancient-disc/blob/main/kb/notebooks/chapters/kinematics_lowfeh.ipynb)

*Third page of the kinematics chapter.* Every number on this page is written by `scripts/paper_text_letter.py` from the tables of
`scripts/kin_feh_lowest.py` (`results/macros_letter.tex`); the code cells are that script, shown but not run (about half an hour),
and `scripts/fig_kin_feh_lowest.py`, which draws the figure from its tables.

The Letter's most metal-poor bin holds the giants with iron abundance [Fe/H] (the logarithm of the iron-to-hydrogen ratio relative
to the Sun's) below -1.5. Its ratio of rotation to vertical velocity dispersion, $v_{\rm rot}/\sigma_z$ (rotation
speed at the radius $R$ = 4 kpc of the disc frame over the dispersion perpendicular to the disc), is 1.8. Do
the stars at the very bottom of that bin, with [Fe/H] below -2, still rotate? The LMC had formed only
$\sim10^{7}$ M☉ of stars when they were born (the formed mass below [Fe/H] = -2, chapter 4), so
their rotation tells us whether the disc existed from the start.

## Why these stars are hard to measure

There are 562 giants with measured [Fe/H] below -2 in the fitted ring, almost all with SDSS-V
BOSS spectra. Their [Fe/H] comes from BOSS, calibrated to APOGEE. Among the stars observed by both surveys, only
40 per cent of those with calibrated BOSS [Fe/H] below -2 have APOGEE [Fe/H] below it;
the others lie at APOGEE [Fe/H] of -1.9 to -1.7. So a group selected by measured [Fe/H] below -2
contains interlopers from just above it. Its purity, the fraction of its stars truly below -2, is uncertain:
the stars in both surveys are brighter than the BOSS sample, and APOGEE itself has few stars below [Fe/H] = -2.2. Milky Way halo
stars, which are metal-poor and kinematically hot, and any hot LMC component add further stars that do not share the disc's
rotation.

## How we measured it

Before running the measurement we asked two independent reviewers to criticise the plan. Both rejected the first idea, to correct
the fitted values of narrow [Fe/H] bins with a matrix of interloper fractions: the bins rotate alike, so the correction has no
leverage, and a fit of a mixture of populations does not return the average of their kinematics. The measurement therefore has
three parts.

1. **Fits of narrow bins.** The kinematic model of chapter 6 (the velocity-ellipsoid likelihood) is fitted to bins of measured
   [Fe/H] inside the Letter's lowest bin, down to [Fe/H] < -2.2. Each bin has its own rotation amplitude, dispersions $\sigma_R$,
   $\sigma_\phi$, $\sigma_z$, Gaia error scale and outlier fraction; everything else (disc orientation, centre, bulk motion,
   streaming and the shape of the rotation curve) is held at the fit of the whole lowest bin, which a few hundred stars cannot
   constrain. Fitting the whole lowest bin this way returns the Letter's value. Holding these parameters is tested by refitting at
   draws of their posterior and with the joint geometry of all populations.
2. **Subsets.** The stars below -2 with the best spectra (BOSS signal-to-noise ratio above 40, or APOGEE), and
   cool and hot giants separately, are fitted the same way.
3. **A null test.** Mock velocities are drawn for the same stars, at their positions and with their measurement errors, from the
   ellipsoid model: a fraction (the purity) are truly metal-poor stars with a chosen $v_{\rm rot}/\sigma_z$, from zero to the
   Letter's value, the rest are interlopers moving like the stars at -1.75 to -1.5,
   and in half of the mocks 15 per cent of the stars are replaced by a hot, non-rotating component with a
   dispersion of 100 km/s, standing in for halo stars. Each mock is
   fitted exactly as the data.

A test of the mock generator (`tests/test_kin_feh_lowest.py`) checks that a fit to a large mock returns the ratio and dispersion
it was drawn with.

## Result

The giants below [Fe/H] = -2 rotate: the two narrow bins below it have
$v_{\rm rot}/\sigma_z\approx$1.5–1.7 (Figure 7.12a), and the
best-measured subset agrees. In the mocks, a group whose truly metal-poor stars do not rotate gives a lower value than the one
measured unless they make up less than 20 per cent of it (Figure 7.12b), and a hot
component lowers the mock values further. For the purities suggested by the stars in both surveys, the truly metal-poor stars
must themselves have $v_{\rm rot}/\sigma_z$ well above 1.

Two points remain open. The ratio falls slightly towards the lowest [Fe/H], which could be real or a sign of a hot component. And
the cool giants rotate faster relative to their dispersion than the hot ones, for reasons we have not established. Neither changes
the conclusion that the stars formed when the LMC had $\sim10^{7}$ M☉ of stars rotate; the Letter reports the
result with its purity caveat.

### The fits and the null test

<details><summary>About <code>scripts/kin_feh_lowest.py</code></summary>

```text
Diagnostic: do the LMC's most metal-poor giants ([Fe/H] < -2) rotate, given that BOSS [Fe/H] errors mix in stars from
-2.0..-1.7 and that Milky Way halo stars and any hot LMC component enter the sample?

The paper's v_rot(4 kpc) / sigma_z of the measured [Fe/H] < -1.5 bin (1.76; scripts/paper_kin_support_ellipsoid.py) comes from
the ellipsoid model of lmcold.kinfit fitted to that bin alone. Here the same likelihood is fitted to narrower bins of measured
[Fe/H] inside it. With a few hundred stars the geometry cannot be fitted per bin, so everything except each bin's own rotation
amplitude v0, dispersions sigma_R, sigma_phi, sigma_z, Gaia error scale k and outlier fraction (OWN) is held at the stage-1
posterior of the [Fe/H] < -1.5 bin (data/kin_mcmc_data_ellipsoid_samples.npz): geometry, centre, bulk motion, streaming, and
R0, vslope, dlnsig_dR. Plan and reviews: two independent reviews rejected unfolding the bins with a mixing matrix (the bins are
nearly collinear, the overlap with APOGEE has 23 stars below -2, and a mixture fit does not return mixture moments); instead:

1. NUTS fits of OWN for each measured bin (BINS) and the subsets of SUBSETS; the bin [Fe/H] < -1.5 itself checks that holding
   the rest reproduces the paper's value.
2. Held parameters: OWN refitted (maximum likelihood) at N_GEO draws of the stage-1 posterior (spread added to the errors), and
   once with the joint (stage-2) geometry of the paper.
3. Null test, end to end: mock velocities for the stars of a bin at their own positions and errors, drawn from the ellipsoid
   model (mean and disc-frame covariance of lmcold.kinfit, Gaia and line-of-sight errors, k = 1). A fraction `purity` of the stars
   are true metal-poor stars with v_rot(4 kpc) / sigma_z = `ratio` (dispersions as the interlopers'); the rest are interlopers with
   the kinematics fitted to measured -1.75..-1.5; a fraction `hot` are replaced by a non-rotating component with dispersion
   HOT_SIGMA about the LMC bulk motion (a stand-in for Milky Way halo stars and an LMC spheroid). Each mock is fitted as the
   data (maximum likelihood); N_MOCK realisations per grid point. The observed ratio is compared with the mock distributions.

Reads the release giants (lmcold.samples.giants()) and the posterior samples above. Writes results/kin_feh_lowest.csv (sample,
n, median [Fe/H], v_rot(4 kpc), sigma_z, ratio median and 16th/84th percentiles, spread over the held parameters, ratio with the
joint geometry, convergence) and results/kin_feh_lowest_null.csv (bin, purity, ratio, hot, realisation, fitted ratio); the figure
is scripts/fig_kin_feh_lowest.py. The Letter's numbers for the [Fe/H] < -2 giants come from these tables (paper_text_letter.py).

Run: .venv/bin/python scripts/manifest.py run kin_feh_lowest.py   (about 30 min on 14 cores: 9 NUTS runs, 256 mock fits)
```
</details>

**Data read by `scripts/kin_feh_lowest.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/giants_apogee.parquet` | Zenodo bundle 1 | APOGEE DR17 giants with Gaia DR3 |
| `data/giants_boss.parquet` | Zenodo bundle 1 | SDSS-V DR20 BOSS giants with Gaia DR3, on the APOGEE scale |
| `data/kin_mcmc_data_ellipsoid_samples.npz` | Zenodo bundle 1 | Posterior samples of the ellipsoid kinematic fit, giants 2-9 deg (NUTS) |
| `results/kin_mcmc_data_ellipsoid.csv` | repository | posterior summary of the joint kinematic fit (velocity ellipsoid), 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid` |

*Shown, not run here: the release giants and the posterior samples of the kinematic fit (Zenodo bundle 1); about 30 min (9 NUTS runs and 256 mock fits). The cells below are the script itself; run it from the repository with* `python scripts/kin_feh_lowest.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
Bins of measured [Fe/H] inside the paper's lowest bin, the subsets, the parameters fitted per bin, and the null-test grid.

```python
import numpyro

numpyro.set_host_device_count(4)  # four chains in parallel; before JAX starts

import numpy as np
import pandas as pd

from fit_kinematics_mcmc import COLS, ring
from lmcold import kinfit, samples
from lmcold.constants import KAPPA
from lmcold.sources import DATA, ROOT

# Bins of measured [Fe/H] (APOGEE scale; -9 = no lower edge): four narrow bins inside the paper's lowest bin, all stars below -2,
# and the paper's whole lowest bin (the check that holding the rest reproduces its value).
BINS = [(-9, -2.2), (-2.2, -2.0), (-2.0, -1.75), (-1.75, -1.5), (-9, -2.0), (-9, -1.5)]
SNR_BEST = 40       # BOSS S/N of the best-measured subset
SUBSETS = {'[Fe/H] < -2.0, BOSS S/N > 40 or APOGEE': lambda c: (c['feh'] < -2.0) & ((c['snr'] > SNR_BEST) | (c['survey'] == 'apogee')),
           '[Fe/H] < -2.0, Teff < 4500 K': lambda c: (c['feh'] < -2.0) & (c['teff'] < 4500),
           '[Fe/H] < -2.0, Teff >= 4500 K': lambda c: (c['feh'] < -2.0) & (c['teff'] >= 4500)}
OWN = ('v0', 'sigma_R', 'sigma_phi', 'sigma_z', 'k', 'logit_fout')   # fitted per bin: rotation amplitude, dispersions [km/s], error scale, outliers
N_GEO = 30          # posterior draws of the held parameters
PURITY = (0.2, 0.4, 0.6, 1.0)     # null test: fraction of a bin's stars that are truly metal-poor (overlap estimate 9/23 = 0.39)
RATIO = (0.0, 0.5, 1.0, 1.76)     # their true v_rot(4 kpc) / sigma_z: none, weak, moderate, the paper's value for [Fe/H] < -1.5
HOT = (0.0, 0.15)   # 0.15: mean Milky Way probability f_mw of the measured [Fe/H] < -2 giants (data/giant_foreground.parquet)
HOT_SIGMA = 100.0   # km/s, isotropic
N_MOCK = 4          # realisations per grid point
NULL_BINS = [(-9, -2.2), (-9, -2.0)]   # bins tested with mocks
INTERLOPER_BIN = (-1.75, -1.5)         # bin whose fitted kinematics the interlopers get (they come from just above -2)
```

#### Held parameters and the mock generator
`held` turns one draw (or the median) of the stage-1 posterior of the [Fe/H] < -1.5 bin into a full parameter vector.
`mock` draws velocities from the ellipsoid model at the stars of a catalogue.

```python
def held(z, i=None):
    """Full kinfit parameter vector (33,) from the stage-1 posterior of population 0 in npz z: draw i (flattened over chains), or
    the median if i is None; parameters the run held (the 0 defaults of start(), e.g. m1c) keep kinfit.start() values except
    the ellipsoid ones, which are all sampled."""
    p = kinfit.start()
    # Keys '1|0|<site>': stage 1, population 0; per-population sites end in '_0'. x: all draws of that site, chains flattened.
    for key in z.files:
        if key.startswith('1|0|'):
            x = np.asarray(z[key]).ravel()
            name = key[4:].removesuffix('_0')
            p[kinfit.PARAMS.index(name)] = np.median(x) if i is None else x[i]
    # Terms the paper's ellipsoid model holds at zero: lopsided part, tilted ring, line-of-sight scale, turning plane.
    for n in ('m1c', 'm1s') + kinfit.TILT + ('qlos',) + kinfit.TURN:
        p[kinfit.PARAMS.index(n)] = 0.0
    return p


def mock(cat, q, rng):
    """Mock proper motions and line-of-sight velocities for catalogue dict cat from parameter vector q (ellipsoid model, k = 1,
    no outliers): returns a copy of cat with pmra, pmdec [mas/yr] and vlos [km/s] replaced (vlos stays NaN where it was)."""
    d = kinfit.data(*(cat[k] for k in COLS[:-1]))   # sky bases and Gaia covariances of the stars
    # Model mean at each star: proper motions [mas/yr], line-of-sight velocity [km/s], distance to the disc plane D [kpc].
    mra, mde, mlos, D = (np.asarray(v) for v in kinfit.model(q, d['rhat'], d['ea'], d['ed']))
    C = np.asarray(kinfit.intrinsic_cov(q, d))                      # (N, 3, 3) km^2/s^2, (east, north, line of sight)
    v = np.einsum('nij,nj->ni', np.linalg.cholesky(C), rng.normal(size=(len(D), 3)))   # intrinsic velocity draws [km/s], (N, 3)
    # Gaia proper-motion errors [mas/yr], (N, 2), with their correlation; line-of-sight errors evlos [km/s] below.
    noise = np.einsum('nij,nj->ni', np.linalg.cholesky(np.asarray(d['cov'])), rng.normal(size=(len(D), 2)))
    kD = KAPPA * D                                                  # km/s per mas/yr at the star
    vlos = mlos + v[:, 2] + rng.normal(size=len(D)) * np.asarray(cat['evlos'])
    return dict(cat, pmra=mra + v[:, 0] / kD + noise[:, 0], pmdec=mde + v[:, 1] / kD + noise[:, 1],
                vlos=np.where(np.isfinite(cat['vlos']), vlos, np.nan))


def with_ratio(q, ratio):
    """Copy of q with v0 set so that v_rot(4 kpc) / sigma_z = ratio (R0 and sigma_z of q)."""
    q = q.copy()
    # v_rot(4 kpc) = v0 tanh(R_SIG / R0) (the vslope term vanishes at R_SIG), so v0 = ratio sigma_z / tanh(R_SIG / R0).
    q[kinfit.PARAMS.index('v0')] = ratio * q[kinfit.PARAMS.index('sigma_z')] / np.tanh(kinfit.R_SIG / q[kinfit.PARAMS.index('R0')])
    return q


def ratio_of(p):
    """v_rot(4 kpc) / sigma_z of vector p."""
    g = lambda n: p[..., kinfit.PARAMS.index(n)]  # noqa: E731  (works for one vector (33,) or a stack of draws (M, 33))
    return g('v0') * np.tanh(kinfit.R_SIG / g('R0')) / g('sigma_z')


def ml(cat, p):
    """Maximum likelihood of OWN for catalogue dict cat, the rest held at p."""
    return kinfit.fit(kinfit.data(*(cat[k] for k in COLS[:-1])), free=OWN, p0=p, ellipsoid=True)[0]


def select(cat, keep):
    """Copy of catalogue dict cat (arrays of shape (N,)) keeping the stars where the boolean array keep is True."""
    return {k: np.asarray(v)[keep] for k, v in cat.items()}
```

#### Load the data
The giants of the paper's lowest bin at 2-9 deg and the stage-1 posterior of that bin.

```python
# The giants of the paper's lowest bin (measured [Fe/H] < -1.5): positions [deg], proper motions [mas/yr], line-of-sight
# velocities [km/s], [Fe/H], Teff, S/N, survey; kept at 2 < rho < 9 deg as in the paper's fit (giants have no distance moduli).
lab, cat = samples.giants()[0]
cat = ring(cat | dict(dmu=np.full(len(cat['ra']), np.nan)))
# Posterior samples of the paper's ellipsoid run; population 0 is the lowest bin (checked), whose stage-1 fit sets the held
# parameters: p_ref, its posterior medians, a full kinfit vector (33,).
z = np.load(DATA / 'kin_mcmc_data_ellipsoid_samples.npz')
assert str(z['names'][0]) == f'giants {lab}' == 'giants [Fe/H] < -1.5'
p_ref = held(z)
n_draws = np.asarray(z['1|0|inc']).size   # number of posterior draws (chains x draws)
# The variant with the joint geometry: p_ref with the shared parameters (centre, bulk motion, inclination, line of nodes,
# streaming) replaced by the stage-2 medians of the joint fit of all populations.
joint = pd.read_csv(ROOT / 'results/kin_mcmc_data_ellipsoid.csv').query("stage == 2 and population == 'shared'")
p_joint = p_ref.copy()
for n, v in zip(joint.param, joint['median']):
    p_joint[kinfit.PARAMS.index(n)] = v
```

#### Fits of the bins
NUTS for OWN in each bin and subset, then the spread over the held parameters and the joint-geometry value.

```python
rng = np.random.default_rng(1)
geo = rng.choice(n_draws, N_GEO, replace=False)   # posterior draws at which the held parameters are tested
# The samples to fit: name -> boolean mask over the stars of the lowest bin.
samples_ = {f'{lo} < [Fe/H] < {hi}'.replace('-9 < ', ''): (cat['feh'] >= lo) & (cat['feh'] < hi) for lo, hi in BINS}
samples_ |= {name: f(cat) for name, f in SUBSETS.items()}
rows, fits = [], {}
fixed = tuple(n for n in kinfit.PARAMS if n not in OWN)   # everything else held at p_ref
for name, keep in samples_.items():
    c = select(cat, keep)                         # the sample's stars
    d = kinfit.data(*(c[k] for k in COLS[:-1]))   # the arrays the likelihood needs (sky bases, covariances)
    p0 = ml(c, p_ref)                                                    # maximum likelihood, the start of NUTS
    m = kinfit.sample([d], [p0], fixed, 500, 1000, 4, ellipsoid=True)    # 500 warm-up, 1000 kept draws, 4 chains
    s = m.get_samples()
    # Full parameter vectors, one per posterior draw: the held values of p0 with the sampled OWN inserted.
    draws = np.tile(p0, (len(s['v0_0']), 1))
    for n in OWN:
        draws[:, kinfit.PARAMS.index(n)] = np.asarray(s[f'{n}_0'])
    r = ratio_of(draws)                                                  # v_rot(4 kpc) / sigma_z per draw
    vrot = r * draws[:, kinfit.PARAMS.index('sigma_z')]                  # v_rot(4 kpc) [km/s]
    spread = np.std([ratio_of(ml(c, held(z, i))) for i in geo])          # ratio's scatter over the held parameters
    # Summary row; the draws' columns 29, 9 and 10 are sigma_z, k and logit_fout (order of kinfit.PARAMS).
    # Convergence: largest split Gelman-Rubin r_hat over the sampled parameters, and the number of divergent transitions.
    rhat = max(float(np.max(numpyro.diagnostics.split_gelman_rubin(np.asarray(v)))) for v in m.get_samples(group_by_chain=True).values())
    div = int(np.sum(np.asarray(m.get_extra_fields()['diverging'])))
    rows.append(dict(sample=name, n=int(keep.sum()), feh_med=np.median(c['feh']),
                     vrot4=np.median(vrot), vrot4_p16=np.percentile(vrot, 16), vrot4_p84=np.percentile(vrot, 84),
                     sigma_z=np.median(draws[:, 29]), ratio_z=np.median(r), ratio_z_p16=np.percentile(r, 16),
                     ratio_z_p84=np.percentile(r, 84), held_spread=spread, ratio_z_joint=ratio_of(ml(c, p_joint)),
                     k=np.median(draws[:, 9]), fout=np.median(1 / (1 + np.exp(-draws[:, 10]))), r_hat=rhat, divergences=div))
    fits[name] = p0
    print(rows[-1], flush=True)
t = pd.DataFrame(rows)   # one row per sample (columns in the module docstring)
t.to_csv(ROOT / 'results/kin_feh_lowest.csv', index=False, float_format='%.4f')
```

#### Null test
Mocks of the lowest bins with a purity of true metal-poor stars of given rotation, the rest interlopers, and a hot fraction.

```python
# The interlopers: the fitted kinematics of INTERLOPER_BIN, with true errors (k = 1) and no outliers. The hot component: the
# same frame and bulk motion, no rotation, isotropic dispersion HOT_SIGMA, constant with radius.
q_int = fits[f'{INTERLOPER_BIN[0]} < [Fe/H] < {INTERLOPER_BIN[1]}'].copy()
q_int[kinfit.PARAMS.index('k')], q_int[kinfit.PARAMS.index('logit_fout')] = 1.0, -10.0
q_hot = q_int.copy()
for n, v in dict(v0=0.0, sigma_R=HOT_SIGMA, sigma_phi=HOT_SIGMA, sigma_z=HOT_SIGMA, dlnsig_dR=0.0).items():
    q_hot[kinfit.PARAMS.index(n)] = v
# The grid: for each tested bin, every purity, true ratio and hot fraction, N_MOCK realisations; each row of `null` is one
# fitted mock.
null = []
for lo, hi in NULL_BINS:
    c = select(cat, (cat['feh'] >= lo) & (cat['feh'] < hi))   # the real stars of the bin: positions and errors are kept
    name = f'{lo} < [Fe/H] < {hi}'.replace('-9 < ', '')
    for purity in PURITY:
        for ratio in RATIO:
            for hot in HOT:
                for j in range(N_MOCK):
                    # One seed per grid point and realisation, so every mock can be reproduced on its own.
                    g = np.random.default_rng([j, int(100 * purity), int(100 * ratio), int(100 * hot), int(-10 * lo)])
                    u = g.random(len(c['ra']))
                    # Velocities of every star under each of the three components: truly metal-poor (rotation `ratio`),
                    # interloper, hot.
                    parts = [mock(c, with_ratio(q_int, ratio), g), mock(c, q_int, g), mock(c, q_hot, g)]
                    # Each star's component: hot with probability `hot`; otherwise metal-poor with probability `purity`.
                    who = np.where(u < hot, 2, np.where(g.random(len(u)) < purity, 0, 1))
                    mc = dict(c, **{k: np.choose(who, [p_[k] for p_ in parts]) for k in ('pmra', 'pmdec', 'vlos')})
                    # Fitted exactly as the data (maximum likelihood of OWN, started from the bin's own fit).
                    null.append(dict(bin=name, purity=purity, ratio=ratio, hot=hot, realisation=j, fit_ratio=ratio_of(ml(mc, fits[name]))))
        print(name, purity, 'done', flush=True)
nt = pd.DataFrame(null)   # bin, purity, ratio, hot, realisation, fitted ratio
nt.to_csv(ROOT / 'results/kin_feh_lowest_null.csv', index=False, float_format='%.4f')
```

### The figure

<details><summary>About <code>scripts/fig_kin_feh_lowest.py</code></summary>

```text
Figure of the check of the most metal-poor giants (scripts/kin_feh_lowest.py, which makes the tables): (a) v_rot(4 kpc) / sigma_z
against the median measured [Fe/H] of the narrow bins inside the paper's lowest bin (geometry held at that bin's fit; errors
include the spread over the held parameters), with the paper's five bins fitted on their own (results/kin_support_ellipsoid.csv);
(b) the null test for measured [Fe/H] < -2.0: v_rot / sigma_z fitted to mocks in which a fraction (purity) of the stars are truly
metal-poor with a given rotation and the rest are interlopers from -1.75..-1.5, without (filled) and with (open) a 15 per cent hot,
non-rotating component, against the observed range (grey band, 16th-84th percentiles).

Reads results/kin_feh_lowest.csv, kin_feh_lowest_null.csv and kin_support_ellipsoid.csv. Writes figures/fig_kin_feh_lowest.png.
Diagnostic figure (KB chapter 'The most metal-poor stars'), not in the paper.

Run: .venv/bin/python scripts/fig_kin_feh_lowest.py   (a few seconds)
```
</details>

**Data read by `scripts/fig_kin_feh_lowest.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/kin_feh_lowest.csv` | repository | written by `python scripts/kin_feh_lowest.py` |
| `results/kin_feh_lowest_null.csv` | repository | written by `python scripts/kin_feh_lowest.py` |
| `results/kin_support_ellipsoid.csv` | repository | written by `python scripts/paper_kin_support_ellipsoid.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The bins and the null-test grid are those of scripts/kin_feh_lowest.py.

In [ ]:
import matplotlib

matplotlib.use('Agg')
import numpy as np
import pandas as pd

from kin_feh_lowest import BINS, HOT, PURITY, RATIO
from lmcold.layout import panel_grid
from lmcold.sources import ROOT

#### Load the data and draw the figure
The three tables; then (a) the ratio against the median measured [Fe/H] and (b) the observed ratio of measured [Fe/H] < -2.0
against the null mocks.

In [ ]:
t = pd.read_csv(ROOT / 'results/kin_feh_lowest.csv').set_index('sample')   # one row per fitted sample (bins and subsets)
nt = pd.read_csv(ROOT / 'results/kin_feh_lowest_null.csv')                  # one row per fitted null mock
# The paper's five giant bins, each fitted on its own (stage 1 of the ellipsoid run), for reference.
ref = pd.read_csv(ROOT / 'results/kin_support_ellipsoid.csv').query("population.str.startswith('giants')")
fig, ax = panel_grid(1, 2, width=7.0, top=1.0, hgap=0.75)   # two equal panels, room above for the keys

# (a) The four narrow bins: median and 16th-84th percentiles of v_rot(4 kpc) / sigma_z, the posterior spread combined in
# quadrature with the spread over the held parameters (held_spread).
a = ax[0, 0]
narrow = t.loc[[f'{lo} < [Fe/H] < {hi}'.replace('-9 < ', '') for lo, hi in BINS[:4]]]
err = np.hypot([narrow.ratio_z - narrow.ratio_z_p16, narrow.ratio_z_p84 - narrow.ratio_z], narrow.held_spread.values)
a.errorbar(narrow.feh_med, narrow.ratio_z, err, fmt='o', color='C3', label='narrow bins of measured [Fe/H], geometry held')
a.errorbar(ref.feh_med, ref.ratio_z, [ref.ratio_z - ref.ratio_z_p16, ref.ratio_z_p84 - ref.ratio_z], fmt='s', mfc='none',
           color='k', label='paper bins (each fitted on its own)')
a.set(xlabel='median measured [Fe/H]', ylabel=r'$v_{\rm rot}(4\,{\rm kpc})/\sigma_z$', ylim=(0, None))
a.legend(loc='lower left', bbox_to_anchor=(0, 1.02), fontsize=7, frameon=False)

# (b) The null test for all stars below -2: the measured value as a grey band (16th-84th percentiles), and every fitted mock
# as a point at its purity, shifted sideways by its true ratio (colour) and by the hot fraction (filled: none; open: HOT[-1]).
b = ax[0, 1]
name = '[Fe/H] < -2.0'
o = t.loc[name]
b.axhspan(o.ratio_z_p16, o.ratio_z_p84, color='0.85', label=f'observed, measured {name}')
for jr, ratio in enumerate(RATIO):
    for hot, mk in zip(HOT, ('o', 's')):
        s = nt[(nt.bin == name) & (nt.ratio == ratio) & (nt.hot == hot)]   # the mocks of this true ratio and hot fraction
        # x position: index of the purity on the axis, offset by 0.12 per true ratio and 0.05 for the hot mocks.
        x = s.purity.map({p: i for i, p in enumerate(PURITY)}) + 0.12 * jr - 0.2 + 0.05 * (hot > 0)
        b.plot(x, s.fit_ratio, mk, color=f'C{jr}', ms=3, mfc='none' if hot else f'C{jr}',
               label=f'true ratio {ratio}' + (f', {hot:.0%} hot' if hot else ''))
b.set(xticks=range(len(PURITY)), xticklabels=[str(p) for p in PURITY], xlabel='purity (fraction of stars truly below -2)',
      ylabel=r'fitted $v_{\rm rot}/\sigma_z$ of the mocks', ylim=(0, None))
b.legend(loc='lower left', bbox_to_anchor=(0, 1.02), fontsize=6, frameon=False, ncol=2)
fig.savefig(ROOT / 'figures/fig_kin_feh_lowest.png', dpi=200)

In [ ]:
from IPython.display import Image
Image(filename='figures/fig_kin_feh_lowest.png')

**Figure 7.12.** (a) $v_{\rm rot}(4\,{\rm kpc})/\sigma_z$ against median measured [Fe/H]: narrow bins inside the
Letter's lowest bin, with the geometry held at that bin's fit (red; errors include the spread over the held parameters), and the
Letter's five bins, each fitted on its own (black squares). (b) The null test for measured [Fe/H] < -2:
$v_{\rm rot}/\sigma_z$ fitted to mocks against the purity, for true ratios of the metal-poor stars from zero (blue) to the
Letter's value (red), without (filled) and with (open) a 15 per cent hot component; grey band: the measured value (16th–84th
percentiles).